System Path Setup

In [1]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [2]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee
import pandas as pd
import geopandas as gpd
import folium # For interactive mapping
from pygbif import occurrences # For GBIF data
from configs.regions import kenyan_coast_roi # Your ROI

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID

print("All core libraries imported and GEE initialized.")

Region of Interest for Kenyan Coast defined.
All core libraries imported and GEE initialized.


Define GBIF Query Parameters & Fetch Data

In [5]:
# Cell 3: Define GBIF Query Parameters & Fetch Data (REVISED BBOX FORMAT)
print("--- Fetching Species Occurrence Data from GBIF ---")

# Get the bounding box from your kenyan_coast_roi
# The bounds() method returns a rectangle's coordinates in [west, south, east, north] order.
# We need to explicitly extract min/max lat/lon for GBIF's parameters.
roi_bbox = kenyan_coast_roi.bounds().getInfo()
# The coordinates for a rectangle are typically [west, south, east, north]
west = roi_bbox['coordinates'][0][0][0]
south = roi_bbox['coordinates'][0][0][1]
east = roi_bbox['coordinates'][0][1][0]
north = roi_bbox['coordinates'][0][2][1] # Or roi_bbox['coordinates'][0][1][1] depending on format

# Let's ensure these are correct. GEE `bounds()` gives a list of coordinates
# For a simple rectangle: [[minLon, minLat], [maxLon, minLat], [maxLon, maxLat], [minLon, maxLat], [minLon, minLat]]
# So: min_lon = coords[0][0][0], min_lat = coords[0][0][1]
#     max_lon = coords[0][1][0], max_lat = coords[0][2][1]
# OR, better:
# Use ee.Geometry.Rectangle which directly gives w,s,e,n
bbox_rectangle = kenyan_coast_roi.bounds()
min_lon, min_lat, max_lon, max_lat = bbox_rectangle.getInfo()['coordinates'][0][0][0], \
                                    bbox_rectangle.getInfo()['coordinates'][0][0][1], \
                                    bbox_rectangle.getInfo()['coordinates'][0][2][0], \
                                    bbox_rectangle.getInfo()['coordinates'][0][2][1]

# GBIF API search parameters
limit_records = 5000

print(f"Querying GBIF for occurrences within bounding box (min_lat:{min_lat}, min_lon:{min_lon}, max_lat:{max_lat}, max_lon:{max_lon}).")
print(f"Attempting to fetch up to {limit_records} records.")

# REVISED: Use decimalLatitude and decimalLongitude range parameters directly
gbif_raw_data = occurrences.search(
    decimalLatitude=f"{min_lat},{max_lat}", # GBIF expects min_lat,max_lat
    decimalLongitude=f"{min_lon},{max_lon}", # GBIF expects min_lon,max_lon
    limit=limit_records
)

if gbif_raw_data and gbif_raw_data['results']:
    print(f"Fetched {len(gbif_raw_data['results'])} raw records from GBIF.")
    gbif_df = pd.DataFrame(gbif_raw_data['results'])
    print(f"DataFrame created with {len(gbif_df)} records.")
    # print("GBIF DataFrame head:\n", gbif_df.head())

    # Save raw data to data/raw/
    output_path = os.path.join(project_root, 'data', 'raw', 'gbif_kenya_coastal_occurrences_raw.csv')
    gbif_df.to_csv(output_path, index=False)
    print(f"Raw GBIF data saved to: {output_path}")

else:
    print("No records found for the specified query or an error occurred.")
    gbif_df = pd.DataFrame() # Create empty DataFrame to avoid errors later

--- Fetching Species Occurrence Data from GBIF ---
Querying GBIF for occurrences within bounding box (min_lat:-4.720644703111179, min_lon:39.19874931843351, max_lat:-1.6660673751001476, max_lon:41.5717961934335).
Attempting to fetch up to 5000 records.
Fetched 300 raw records from GBIF.
DataFrame created with 300 records.
Raw GBIF data saved to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\data\raw\gbif_kenya_coastal_occurrences_raw.csv


Initial Processing and Cleaning of GBIF Data

In [8]:
# Cell 4: Initial Processing and Cleaning of GBIF Data (REVISED FOR COLUMN EXISTENCE)
print("--- Cleaning GBIF Occurrence Data ---")

# Load the raw GBIF data
input_path = os.path.join(project_root, 'data', 'raw', 'gbif_kenya_coastal_occurrences_raw.csv')
if os.path.exists(input_path):
    gbif_df = pd.read_csv(input_path, low_memory=False)
    print(f"Loaded {len(gbif_df)} raw GBIF records.")
    print(f"Original columns: {gbif_df.columns.tolist()}") # Print columns for inspection
else:
    print(f"Error: Raw GBIF data not found at {input_path}. Please re-run Cell 3 or check path.")
    sys.exit("No raw GBIF data to process.") # Exit here if no data

if not gbif_df.empty:
    # --- Basic Cleaning Steps ---

    # 1. Remove records without coordinates
    initial_count = len(gbif_df)
    gbif_df.dropna(subset=['decimalLatitude', 'decimalLongitude'], inplace=True)
    print(f"Removed {initial_count - len(gbif_df)} records without coordinates.")

    # 2. Remove records with high coordinate uncertainty (if column exists)
    if 'coordinateUncertaintyInMeters' in gbif_df.columns:
        initial_count = len(gbif_df)
        gbif_df = gbif_df[gbif_df['coordinateUncertaintyInMeters'].fillna(0) <= 1000]
        print(f"Removed {initial_count - len(gbif_df)} records with high coordinate uncertainty (>1000m).")
    else:
        print("Skipping coordinate uncertainty filter: 'coordinateUncertaintyInMeters' column not found.")

    # 3. Remove records with known geospatial issues (if 'issue' column exists)
    # GBIF provides a helpful 'issue' column for data quality flags.
    # We can filter out common issues like 'ZERO_COORDINATE', 'COORDINATE_INVALID', etc.
    if 'issue' in gbif_df.columns:
        initial_count = len(gbif_df)
        # Filter rows where the 'issue' column DOES NOT contain any of the problematic issues
        gbif_df = gbif_df[~gbif_df['issue'].str.contains('COORDINATE_INVALID|ZERO_COORDINATE', na=False)]
        print(f"Removed {initial_count - len(gbif_df)} records with specific geospatial issues (from 'issue' column).")
    else:
        print("Skipping geospatial issues filter: 'issue' column not found.")


    # 4. Remove duplicate records (based on key identifying columns)
    initial_count = len(gbif_df)
    # Ensure all columns in subset actually exist before dropping duplicates
    unique_subset_cols = ['scientificName', 'eventDate', 'decimalLatitude', 'decimalLongitude']
    existing_subset_cols = [col for col in unique_subset_cols if col in gbif_df.columns]
    
    if len(existing_subset_cols) == len(unique_subset_cols):
         gbif_df.drop_duplicates(subset=existing_subset_cols, inplace=True)
         print(f"Removed {initial_count - len(gbif_df)} duplicate records based on: {', '.join(existing_subset_cols)}.")
    else:
        print(f"Skipping duplicate removal: Not all required columns for subsetting found. Missing: {list(set(unique_subset_cols) - set(existing_subset_cols))}")


    # 5. Convert to GeoDataFrame for easier spatial operations later
    # Ensure latitude/longitude columns still exist after previous filters
    if 'decimalLongitude' in gbif_df.columns and 'decimalLatitude' in gbif_df.columns:
        gbif_gdf = gpd.GeoDataFrame(
            gbif_df,
            geometry=gpd.points_from_xy(gbif_df.decimalLongitude, gbif_df.decimalLatitude),
            crs="EPSG:4326"
        )
        print(f"Cleaned and converted to GeoDataFrame with {len(gbif_gdf)} records.")
        # Save processed data to data/processed/
        output_processed_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_coastal_occurrences_cleaned.geojson')
        gbif_gdf.to_file(output_processed_path, driver='GeoJSON')
        print(f"Cleaned GBIF data saved to: {output_processed_path}")
    else:
        print("Cannot create GeoDataFrame: 'decimalLatitude' or 'decimalLongitude' columns not found after filtering.")
        gbif_gdf = gpd.GeoDataFrame() # Create empty for consistency
else:
    print("No GBIF data to clean after initial load.")
    gbif_gdf = gpd.GeoDataFrame() # Create empty for consistency

--- Cleaning GBIF Occurrence Data ---
Loaded 300 raw GBIF records.
Original columns: ['key', 'datasetKey', 'publishingOrgKey', 'installationKey', 'hostingOrganizationKey', 'publishingCountry', 'protocol', 'lastCrawled', 'lastParsed', 'crawlId', 'extensions', 'basisOfRecord', 'occurrenceStatus', 'lifeStage', 'classifications', 'taxonKey', 'kingdomKey', 'phylumKey', 'classKey', 'orderKey', 'familyKey', 'genusKey', 'speciesKey', 'acceptedTaxonKey', 'scientificName', 'scientificNameAuthorship', 'acceptedScientificName', 'kingdom', 'phylum', 'order', 'family', 'genus', 'species', 'genericName', 'specificEpithet', 'taxonRank', 'taxonomicStatus', 'iucnRedListCategory', 'dateIdentified', 'decimalLatitude', 'decimalLongitude', 'coordinateUncertaintyInMeters', 'continent', 'stateProvince', 'gadm', 'year', 'month', 'day', 'eventDate', 'startDayOfYear', 'endDayOfYear', 'issues', 'modified', 'lastInterpreted', 'references', 'license', 'isSequenced', 'identifiers', 'media', 'facts', 'relations', 'is

INFO:Created 209 records


Cleaned GBIF data saved to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\data\processed\gbif_kenya_coastal_occurrences_cleaned.geojson
